# 01 — Corpus Health

Data quality audit: nulls, duplicates, distributions, outliers, coverage.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

conn = get_connection(read_only=False)
init_schema(conn)
corpus = build_feature_matrix(conn)

print(f"Corpus: {corpus.shape[0]:,} rows × {corpus.shape[1]} cols")
print(f"Estimated memory: {corpus.estimated_size('mb'):.1f} MB")

## 1. Schema Overview

In [ ]:
# Dtypes and non-null counts
schema_df = pl.DataFrame({
    "column": corpus.columns,
    "dtype": [str(d) for d in corpus.dtypes],
    "null_count": [corpus[c].null_count() for c in corpus.columns],
    "null_pct": [round(corpus[c].null_count() / len(corpus) * 100, 2) for c in corpus.columns],
})
schema_df.sort("null_pct", descending=True).head(20)

## 2. Null Audit

In [ ]:
# Columns with >0% nulls
cols_with_nulls = schema_df.filter(pl.col("null_pct") > 0).sort("null_pct", descending=True)
print(f"{len(cols_with_nulls)} columns have nulls:")
cols_with_nulls

In [ ]:
# Null heatmap (top 20 most-null columns, sampled rows)
if len(cols_with_nulls) > 0:
    top_null_cols = cols_with_nulls["column"].to_list()[:20]
    sample = corpus.select(top_null_cols).sample(n=min(500, len(corpus)), seed=42)
    null_matrix = sample.select([pl.col(c).is_null().cast(pl.Int8) for c in top_null_cols]).to_numpy()

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(null_matrix.T, cbar=False, yticklabels=top_null_cols, xticklabels=False, ax=ax, cmap="YlOrRd")
    ax.set_title("Null Pattern (yellow=present, red=null)")
    plt.tight_layout()
    plt.show()
else:
    print("No nulls found — corpus is clean.")

## 3. `features_source` Provenance

In [ ]:
if "features_source" in corpus.columns:
    source_counts = corpus.group_by("features_source").len().sort("len", descending=True)
    print(source_counts)

    fig, ax = plt.subplots(figsize=(8, 4))
    labels = source_counts["features_source"].to_list()
    values = source_counts["len"].to_list()
    ax.barh(labels, values)
    ax.set_xlabel("Track count")
    ax.set_title("Audio Feature Provenance")
    plt.tight_layout()
    plt.show()
else:
    print("No features_source column")

## 4. Duplicates

In [ ]:
# Exact ID duplicates
n_unique_ids = corpus["id"].n_unique()
n_total = len(corpus)
print(f"Total rows: {n_total:,}")
print(f"Unique IDs: {n_unique_ids:,}")
print(f"Duplicate IDs: {n_total - n_unique_ids:,}")

# Near-duplicates: same track_name + first_genre (potential re-releases)
if "track_name" in corpus.columns:
    name_dupes = (
        corpus.group_by("track_name")
        .agg(pl.col("id").n_unique().alias("n_ids"))
        .filter(pl.col("n_ids") > 1)
        .sort("n_ids", descending=True)
    )
    print(f"\nTrack names with multiple IDs: {len(name_dupes)}")
    name_dupes.head(10)

## 5. Audio Feature Distributions

In [ ]:
AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]

available_audio = [f for f in AUDIO_FEATURES if f in corpus.columns]
n_feats = len(available_audio)
ncols = 3
nrows = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = axes.flatten()

for i, feat in enumerate(available_audio):
    values = corpus[feat].drop_nulls().to_numpy()
    axes[i].hist(values, bins=50, edgecolor="white", alpha=0.8)
    axes[i].set_title(feat)
    axes[i].axvline(np.median(values), color="red", linestyle="--", alpha=0.7, label=f"median={np.median(values):.2f}")
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Audio Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Additional: popularity, fave_score, n_playlists
extra_features = ["popularity", "fave_score", "n_playlists"]
available_extra = [f for f in extra_features if f in corpus.columns]

fig, axes = plt.subplots(1, len(available_extra), figsize=(5 * len(available_extra), 4))
if len(available_extra) == 1:
    axes = [axes]

for i, feat in enumerate(available_extra):
    values = corpus[feat].drop_nulls().to_numpy()
    axes[i].hist(values, bins=50, edgecolor="white", alpha=0.8)
    axes[i].set_title(feat)

plt.tight_layout()
plt.show()

### 5b. Key & Mode Distribution

Musical key and major/minor breakdown across the library.

In [ ]:
# Key and mode distributions from DuckDB (key_labels, mode_labels, key_mode)
key_mode_df = conn.execute("""
    SELECT key_labels, mode_labels, key_mode
    FROM track_profile
    WHERE key_labels IS NOT NULL AND key_labels != ''
""").pl()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Key distribution
key_counts = key_mode_df.group_by("key_labels").len().sort("len", descending=True)
axes[0].bar(key_counts["key_labels"].to_list(), key_counts["len"].to_list())
axes[0].set_title("Tracks by Musical Key")
axes[0].tick_params(axis="x", rotation=45)

# Major vs Minor
mode_counts = key_mode_df.group_by("mode_labels").len().sort("len", descending=True)
axes[1].bar(mode_counts["mode_labels"].to_list(), mode_counts["len"].to_list())
axes[1].set_title("Major vs Minor")

# Top key_mode combos
km_counts = key_mode_df.group_by("key_mode").len().sort("len", descending=True).head(15)
axes[2].barh(km_counts["key_mode"].to_list()[::-1], km_counts["len"].to_list()[::-1])
axes[2].set_title("Top 15 Key + Mode Combos")

plt.tight_layout()
plt.show()

print(f"Total tracks with key data: {len(key_mode_df):,}")
print(f"Keys: {key_counts.height}, Modes: {mode_counts.height}")

## 6. ENOA Coordinate Coverage

In [ ]:
if "top" in corpus.columns and "left" in corpus.columns:
    has_enoa = corpus.filter((pl.col("top") != 0) | (pl.col("left") != 0))
    no_enoa = corpus.filter((pl.col("top") == 0) & (pl.col("left") == 0))
    print(f"Tracks with ENOA coords: {len(has_enoa):,} ({len(has_enoa)/len(corpus)*100:.1f}%)")
    print(f"Tracks at (0,0) / missing:  {len(no_enoa):,} ({len(no_enoa)/len(corpus)*100:.1f}%)")

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(
        has_enoa["left"].to_numpy(),
        has_enoa["top"].to_numpy(),
        s=1, alpha=0.3, label=f"has coords ({len(has_enoa):,})"
    )
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.set_title("ENOA Coverage — Library Tracks")
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No ENOA columns in corpus")

## 7. Outlier Detection

In [ ]:
outlier_features = ["tempo", "loudness", "duration_ms"]
available_outlier = [f for f in outlier_features if f in corpus.columns]

for feat in available_outlier:
    values = corpus[feat].drop_nulls()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = values.filter((values < lower) | (values > upper)).len()
    print(f"{feat:20s}  IQR=[{q1:.1f}, {q3:.1f}]  bounds=[{lower:.1f}, {upper:.1f}]  outliers={n_outliers} ({n_outliers/len(values)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, len(available_outlier), figsize=(5 * len(available_outlier), 4))
if len(available_outlier) == 1:
    axes = [axes]

for i, feat in enumerate(available_outlier):
    values = corpus[feat].drop_nulls().to_numpy()
    axes[i].boxplot(values, vert=True)
    axes[i].set_title(feat)

plt.suptitle("Outlier Box Plots")
plt.tight_layout()
plt.show()

### 7b. Per-Playlist Centroid Outliers

For each playlist, compute the audio-feature centroid, then measure each track's
z-scored Euclidean distance from that centroid. Surfaces tracks that are
stylistically unusual for their playlist.

In [ ]:
# Join corpus tracks with playlist membership via DuckDB
OUTLIER_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]
available_of = [f for f in OUTLIER_FEATURES if f in corpus.columns]

track_playlists = conn.execute("""
    SELECT pt.track_id, p.playlist_name
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
""").pl()

# Merge corpus features with playlist membership (one row per track×playlist)
track_features = corpus.select(["id", "track_name"] + available_of).rename({"id": "track_id"})
per_playlist = track_playlists.join(track_features, on="track_id", how="inner")

# Compute per-playlist centroid and std
centroids = per_playlist.group_by("playlist_name").agg(
    [pl.col(f).mean().alias(f"{f}_mean") for f in available_of]
    + [pl.col(f).std().alias(f"{f}_std") for f in available_of]
)

# Join centroids back, compute z-scored Euclidean distance
scored = per_playlist.join(centroids, on="playlist_name")
z_components = []
for feat in available_of:
    std_col = f"{feat}_std"
    mean_col = f"{feat}_mean"
    z_components.append(
        ((pl.col(feat) - pl.col(mean_col)) / pl.col(std_col).clip(lower_bound=1e-9)).pow(2)
    )

scored = scored.with_columns(
    pl.sum_horizontal(z_components).sqrt().alias("outlier_score")
)

# Top-20 most outlying tracks
top_outliers = (
    scored.select(["track_name", "playlist_name", "outlier_score"])
    .sort("outlier_score", descending=True)
    .head(20)
)
print("Top 20 per-playlist centroid outliers (z-score distance):")
top_outliers

In [ ]:
# KDE histogram: outlier score distribution by playlist
playlists = scored["playlist_name"].unique().sort().to_list()

fig, ax = plt.subplots(figsize=(14, 6))
for pname in playlists:
    values = scored.filter(pl.col("playlist_name") == pname)["outlier_score"].to_numpy()
    if len(values) > 2:
        sns.kdeplot(values, ax=ax, label=pname, fill=True, alpha=0.15, linewidth=1.5)

ax.set_xlabel("Outlier Score (z-score distance from playlist centroid)")
ax.set_title("Per-Playlist Outlier Score Distribution")
ax.legend(fontsize=8, ncol=2, loc="upper right")
plt.tight_layout()
plt.show()

# Summary stats per playlist
playlist_summary = (
    scored.group_by("playlist_name")
    .agg(
        pl.col("outlier_score").mean().alias("mean_score"),
        pl.col("outlier_score").median().alias("median_score"),
        pl.col("outlier_score").max().alias("max_score"),
        pl.len().alias("n_tracks"),
    )
    .sort("mean_score", descending=True)
)
print("Outlier score summary per playlist:")
playlist_summary

### 7c. Outlier ENOA Scatter

Where do per-playlist outliers sit in ENOA genre space?

In [ ]:
# Plot outliers in ENOA space per playlist — need top/left from corpus
if "top" in corpus.columns and "left" in corpus.columns:
    # Add ENOA coords to scored DataFrame
    enoa_cols = corpus.select(["id", "top", "left"]).rename({"id": "track_id"})
    scored_enoa = scored.join(enoa_cols, on="track_id", how="inner")

    # Flag outliers as top-5% by score within each playlist
    scored_enoa = scored_enoa.with_columns(
        (pl.col("outlier_score") > pl.col("outlier_score").quantile(0.95).over("playlist_name"))
        .alias("is_outlier")
    )

    unique_playlists = scored_enoa["playlist_name"].unique().sort().to_list()
    ncols_enoa = 4
    nrows_enoa = (len(unique_playlists) + ncols_enoa - 1) // ncols_enoa

    fig, axes = plt.subplots(nrows_enoa, ncols_enoa, figsize=(16, 4 * nrows_enoa))
    axes = axes.flatten()

    for i, pname in enumerate(unique_playlists):
        pdata = scored_enoa.filter(pl.col("playlist_name") == pname)
        normal = pdata.filter(~pl.col("is_outlier"))
        outliers_p = pdata.filter(pl.col("is_outlier"))
        axes[i].scatter(normal["left"].to_numpy(), normal["top"].to_numpy(), s=5, alpha=0.3, label="normal")
        axes[i].scatter(outliers_p["left"].to_numpy(), outliers_p["top"].to_numpy(), s=15, alpha=0.8, color="red", label="outlier")
        axes[i].set_title(pname, fontsize=9)
        axes[i].invert_yaxis()

    for j in range(len(unique_playlists), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Per-Playlist Outliers in ENOA Space (red = top 5%)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No ENOA coordinates in corpus")

## 8. Decade / Year Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if "decade" in corpus.columns:
    decade_counts = corpus.group_by("decade").len().sort("decade")
    axes[0].bar(decade_counts["decade"].to_list(), decade_counts["len"].to_list())
    axes[0].set_title("Tracks per Decade")
    axes[0].tick_params(axis="x", rotation=45)

if "year" in corpus.columns:
    years = corpus["year"].drop_nulls().to_numpy()
    axes[1].hist(years, bins=50, edgecolor="white", alpha=0.8)
    axes[1].set_title("Release Year Distribution")

plt.tight_layout()
plt.show()

### 8b. Median Audio Features by Decade

How has the library's sonic profile shifted across decades?

In [ ]:
# Median audio features by decade — heatmap
if "decade" in corpus.columns:
    decade_data = corpus.filter(pl.col("decade").is_not_null()).select(["decade"] + available_audio)
    decade_medians = (
        decade_data.group_by("decade")
        .agg([pl.col(f).median().alias(f) for f in available_audio])
        .sort("decade")
    )

    # Convert to numpy for seaborn heatmap
    decades = decade_medians["decade"].to_list()
    heatmap_data = decade_medians.select(available_audio).to_numpy()

    fig, ax = plt.subplots(figsize=(12, max(4, len(decades) * 0.5)))
    sns.heatmap(
        heatmap_data,
        xticklabels=available_audio,
        yticklabels=decades,
        annot=True, fmt=".2f", cmap="coolwarm",
        ax=ax, linewidths=0.5,
    )
    ax.set_title("Median Audio Features by Decade")
    plt.tight_layout()
    plt.show()
else:
    print("No decade column in corpus")

## 9. Embedding Coverage

In [ ]:
t2v_cols = [c for c in corpus.columns if c.startswith("t2v_")]
if t2v_cols:
    # A row has embeddings if any t2v column is non-zero
    has_embedding = corpus.select(
        pl.any_horizontal([pl.col(c) != 0 for c in t2v_cols]).alias("has_emb")
    )["has_emb"]
    n_with = has_embedding.sum()
    print(f"Track2Vec dims: {len(t2v_cols)}")
    print(f"Tracks with embeddings: {n_with:,} / {len(corpus):,} ({n_with/len(corpus)*100:.1f}%)")
else:
    print("No t2v_* columns found")

## 10. Summary Statistics

In [ ]:
# Numeric summary
numeric_cols = [c for c in corpus.columns if corpus[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)]
# Limit to key features
key_numeric = [c for c in AUDIO_FEATURES + ["popularity", "fave_score", "n_playlists", "year_normalized", "playlist_diversity"] if c in numeric_cols]
corpus.select(key_numeric).describe()

In [ ]:
conn.close()